# Snowflake Sampling

<a id="topics"></a>
### Topics in this lesson


1. [Initial setup](#Initial_setup)
1. [Sampling](#Sampling)  
    

<a id="Initial_setup"></a>
## 1. Initial Setup

#### Connect and create a `Session`

The following cell connects to your Snowflake account and creates an instance of `Session`. 

*You needn't modify anything in this cell. Just run it.*
> &#10071; Success requires that you have already completed the key pair authentication exercise.

In [ ]:
# Run utils notebook
%run ../../utils/ds_utils_python.ipynb

# Connect to Snowflake and create a Session object named session
session = create_session()

---
#### Setup Code

We should set ourselves up for success. The following ensures our context is set properly for database, schema, role, and warehouse.

*You needn't edit anything in the following cell. Just run it.*

In [ ]:
# Hard code the lesson name
lesson_name = "SNOWPARK_SAMPLING_PY"

# Create the context items for this lesson
lesson = confirm_or_create_lesson_context(session, lesson_name)

<a id="Sampling"></a>
## 2. Sampling

**Taking a Representative sample of a data set**

- Preferred over RANDOM or LIMIT as a sampling method


In Snowpark we have a `snowflake.snowpark.Table` class, it has the method `Sample`

This Sample method samples rows based on either the number of rows to be returned or a percentage of rows to be returned.



The sample method comes with the following arguments:
    
    
    Parameters
- **frac** - frac – The percentage of rows to be sampled.

- **n** - The fixed number of rows to sample in the range of 0 to 1,000,000 (inclusive). Either frac or n should be provided.

- **seed** - Specifies a seed value to make the sampling deterministic. Can be any integer between 0 and 2147483647 inclusive. Default value is None.

- **sampling_method** - Specifies the sampling method to use: - “BERNOULLI” / “ROW” or "BLOCK" / "SYSTEM"


Let's have a small example:

In [ ]:
# We sample from a table called customer_loyalty
# full_table_name = "tasty_bytes.raw_customer.customer_loyalty"
full_table_name = "TRAINING_DB.TPCH_SF10.lineitem"
table = session.table(full_table_name)

In [ ]:
# This table contains about 60 million rows
table.count()

As both `Dataframe` and `Table` have a sampling method. We check the type to make sure we are still using a `table`

In [ ]:
print(f"The class of table is {type(table)}")

In [ ]:
sample1 = table.sample(0.01)
sample1.show(5)

sample1.count()

> &#10071; Notice it return 1% of the rows. This is the proportion, not the percentage

In [ ]:
# We can also sample a number of rows instead of a proportion
sample1 = table.sample(n = 500000)
sample1.show(5)

sample1.count()

We are not specifying a sampling method as it takes the default. 

The default is None so then the Snowflake database will use “ROW” by default.

Let's do some sampling where we specify Bernoulli 

In [ ]:
# We can also sample a number of rows instead of a proportion
sample1 = table.sample(n = 500000, sampling_method = "BERNOULLI")
sample1.show(5)

sample1.count()


As we can see, there is no difference between None, Row and Bernoulli as they all refer to the same sampling method.

Let's do some SYSTEM sampling now

In [ ]:
# We can't SYSTEM sample 1 million rows as sampling with a fixed size is not supported for SYSTEM sampling
# This will error:
#sample1 = table.sample(n = 1000000, sampling_method = "SYSTEM")

# We will sample a percentage instead
sample1 = table.sample(0.1, sampling_method = "SYSTEM")

sample1.show(5)

sample1.count()

Notice that the sampling amount is now less accurate. This is because it now uses whole MicroPartitions

#### Seed

The last thing we can set when sampling is a seed.

This is to make the results deterministic

In [ ]:
# We can also sample a number of rows instead of a proportion
sample1 = table.sample(0.2, sampling_method = "BERNOULLI", seed = 400)
sample1.show(5)

sample1.count()

> &#10071; A seed can't be set when sampling a specific number of rows using n

In [ ]:
close_session_and_clean_up(get_lesson())

### &#10071; `Shut Down Kernel`
> After completing the activities in a notebook and before moving on to the next exercise, shut down the completed notebook by right-clicking on the notebook name and selecting `Shut Down Kernel`.